<a href="https://colab.research.google.com/github/IrumShehryar/ML-NLP-Coursework/blob/main/nlp/04-neural-network/markov-model-for-text-generation/project05-ThirdOrder_markov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
import numpy as np
import string # for creating string

# Get the data and tokenize it

In [2]:
text = " The quick brown fox jumps over the lazy dog and the quick white cat jumps over the lazy rabbit"

In [3]:
tokens = text.lower().split()

In [4]:
T = len(tokens) # Number of states in Markov Model

# Initialize the distribution dictionaries

In [5]:
pi = {} # Initial distribution representing the start of the sentencee
A1 = {} # First order transition for second word only. it is different from the A used for text classification.
A2 = {} # This will store second order transitions i.e the words after the second word.
A3 = {} # This will store third order transitions i.e the words after the second word.

# Create the function that will fill the above dictionaries

In [6]:
def fill_dict(d, k, v): # Three inputs are dictionary, keys and values. Keys represent some starting words or pair
                        #of words in the second order case as shown in above figure. First we collect the words from text
                        # and then assign prob to the words.
  if k not in d:
    d[k] = []
  d[k].append(v)  # We are collecting the phrases here

# Iterate over the text to fill the dictionaries

In [7]:
for i in range(T):
    t = tokens[i]    # Grab the ith token
    if i == 0:       # This is the first word in sentence so we would like to update our first word in distribution.
                     # we can do it by incrementing the value stored in t by one

      pi[t] = pi.get(t, 0.) + 1
    else:
      t_1 = tokens[i-1]
    if i == 1:
        fill_dict(A1, t_1, t)
    else:
      t_2 = tokens[i-2]

    if i == 2:
        fill_dict(A2,(t_2, t_1), t)
    else:
        t_3 = tokens[i-3]
    if (i!=0) and (i!=1) and (i!=2):
        fill_dict(A3, (t_3, t_2, t_1), t)
    if i == T - 1:
        fill_dict(A3, (t_2, t_1, t), '<end>')




# Observe the distributions

In [8]:
pi

{'the': 1.0}

In [9]:
A1 # it contains the second word as a value and first word as a key.

{'the': ['quick']}

In [10]:
A2# it contains two previous words as keys and current word as value

{('the', 'quick'): ['brown']}

In [11]:
A3

{('the', 'quick', 'brown'): ['fox'],
 ('quick', 'brown', 'fox'): ['jumps'],
 ('brown', 'fox', 'jumps'): ['over'],
 ('fox', 'jumps', 'over'): ['the'],
 ('jumps', 'over', 'the'): ['lazy', 'lazy'],
 ('over', 'the', 'lazy'): ['dog', 'rabbit'],
 ('the', 'lazy', 'dog'): ['and'],
 ('lazy', 'dog', 'and'): ['the'],
 ('dog', 'and', 'the'): ['quick'],
 ('and', 'the', 'quick'): ['white'],
 ('the', 'quick', 'white'): ['cat'],
 ('quick', 'white', 'cat'): ['jumps'],
 ('white', 'cat', 'jumps'): ['over'],
 ('cat', 'jumps', 'over'): ['the'],
 ('the', 'lazy', 'rabbit'): ['<end>']}

# Create a function to convert list of possible words to dictionary of probabilities

In [12]:
def list_to_probdict(tokens): # This function does two things. First it creates the dictionary of count and then normalize
                    # the counts to convert each count into probability. The input to the function is ts which is
                    # list of the token
  d = {}
  n = len(tokens)    # The total number of samples which is length of ts
  for t in tokens:   # loop through each token
    d[t] = d.get(t, 0.) + 1     # increment the value by 1 each time we encounter the token. After this for loop we have the dictionary
                                # of counts where key is the token and value is the corressponding count
  for t, c in d.items():
    d[t] = c / n
  return d

# Apply the function to first order and second order transitions

In [13]:
# Now we have a first order dictionary which stores second word of each sentence. we have previously stored in this
# dictionery is the list of possible next tokens. we replace the list of tokens by dictionary of probabilities
# by applting the function list2pdict
for t_1, token in A1.items():
  # replace list with dictionary of probabilities
  A1[t_1] = list_to_probdict(token)

In [14]:
A1

{'the': {'quick': 1.0}}

In [15]:
for (t_2, t_1), token in A2.items():
  A2[(t_2, t_1)] = list_to_probdict(token)

In [16]:
A2

{('the', 'quick'): {'brown': 1.0}}

In [17]:
for (t_3, t_2, t_1), token in A3.items():
  A3[(t_3, t_2, t_1)] = list_to_probdict(token)

In [18]:
A3

{('the', 'quick', 'brown'): {'fox': 1.0},
 ('quick', 'brown', 'fox'): {'jumps': 1.0},
 ('brown', 'fox', 'jumps'): {'over': 1.0},
 ('fox', 'jumps', 'over'): {'the': 1.0},
 ('jumps', 'over', 'the'): {'lazy': 1.0},
 ('over', 'the', 'lazy'): {'dog': 0.5, 'rabbit': 0.5},
 ('the', 'lazy', 'dog'): {'and': 1.0},
 ('lazy', 'dog', 'and'): {'the': 1.0},
 ('dog', 'and', 'the'): {'quick': 1.0},
 ('and', 'the', 'quick'): {'white': 1.0},
 ('the', 'quick', 'white'): {'cat': 1.0},
 ('quick', 'white', 'cat'): {'jumps': 1.0},
 ('white', 'cat', 'jumps'): {'over': 1.0},
 ('cat', 'jumps', 'over'): {'the': 1.0},
 ('the', 'lazy', 'rabbit'): {'<end>': 1.0}}

# Create a function for sampling the words

In [19]:
def sample_word(d): #Here input "d" is the dictionary of probability. key is the possible word and value is the corresponding probability
  p0 = np.random.random() # draw a sample from uniform distribution
  cumulative = 0
  for t, p in d.items():
    cumulative += p
    if p0 < cumulative:
      return t

# Create a function to generate the text

In [20]:
def generateText():
  for i in range(5): # generate 5 lines
    sentence = []

    # sample initial word
    w0 = sample_word(pi)
    sentence.append(w0)

    # sample second word
    w1 = sample_word(A1[w0])
    sentence.append(w1)

    # sample third word
    w2 = sample_word(A2[(w0,w1)])
    sentence.append(w2)

    # Third-order transitions until <end>
    while True:
      w3 = sample_word(A3[(w0, w1, w2)])
      if w3 == '<end>':
        break
      sentence.append(w3)
      w0 = w1 # update the previous word
      w1 = w2 # update the previous word
      w2 = w3
    print(' '.join(sentence))

# Generate five lines of text

In [23]:
generateText()

the quick brown fox jumps over the lazy dog and the quick white cat jumps over the lazy rabbit
the quick brown fox jumps over the lazy rabbit
the quick brown fox jumps over the lazy dog and the quick white cat jumps over the lazy dog and the quick white cat jumps over the lazy dog and the quick white cat jumps over the lazy dog and the quick white cat jumps over the lazy dog and the quick white cat jumps over the lazy dog and the quick white cat jumps over the lazy dog and the quick white cat jumps over the lazy rabbit
the quick brown fox jumps over the lazy rabbit
the quick brown fox jumps over the lazy rabbit
